# 10.10 配對標籤怎麼決定分數的調整方向？


兩張圖的正確描述依序放在文字欄 0、1。希望每張圖的正確欄分數比另一欄高。將同批文字當候選、用正確欄當標籤，就能算猜錯代價；這種拿正確配對和其他候選比較的學習稱對比學習。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch.nn import functional as F

scores = torch.tensor([[2.0, 0.0], [0.0, 2.0]], requires_grad=True)
targets = torch.arange(2)
loss = F.cross_entropy(scores / 0.5, targets)
loss.backward()
print("代價", round(loss.item(), 4))
print("梯度", scores.grad)

本例手設分數 `[[2,0],[0,2]]`，並不等於上一節的餘弦矩陣。`targets` 為 `[0,1]`；分數除以正數 0.5 是溫度縮放，使候選差距放大。交叉熵比較正確欄機率，代價約 0.0181，梯度約 `[[-0.0180,0.0180],[0.0180,-0.0180]]`。

梯度下降用舊值減去步長乘梯度。正確欄是負梯度，若以 0.1 更新，2 約成 2.0018；錯誤欄是正梯度，0 約成 −0.0018。這是從符號追到調整方向的手算，程式只做 backward，沒有真正更新。

若分數由圖片和文字編碼器算出，梯度會沿計算傳回兩端。共享參數會同時影響多個分數，因此更新後需要重新算矩陣，不能只在報表對角線加數字就算完成訓練。

還要先確認候選確實不配。兩張圖若都是紅圓、兩段描述也都是「紅圓」，非對角欄同樣正確；硬把它當錯誤便是假負例。可以允許多個正確描述，或在入門批次先選不同屬性的配對。

練習將標籤改為 `[1,0]`，預期非對角欄收到負梯度，代價提高。這是把配對告訴模型反了，不是合理的資料增強。

<details>
<summary>回顧與查證</summary>

可回顧：[10.9相似度矩陣](https://birdhackor.github.io/tiny-perceptron-vlm/10.9.html)、[1.8交叉熵](https://birdhackor.github.io/tiny-perceptron-vlm/1.8.html)、[1.12梯度下降更新](https://birdhackor.github.io/tiny-perceptron-vlm/1.12.html)。

原實驗的資料切分、更新設定與逐題結果見[既有實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/contrastive.json)；重做入口見[實驗說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/README.md)。這是上述有限任務的歷史紀錄。

</details>
